In [ ]:
import numpy as np
import pandas as pd
import os

filename = "large_dataset.csv"
target_size_gb = 5.2  # حجم أكبر من 5 جيجا
chunk_rows = 1_000_000

# إنشاء الملف وكتابة الرأس (Header)
df_head = pd.DataFrame({
    "id": np.arange(10),
    "feature_1": np.random.randn(10),
    "feature_2": np.random.randint(0, 100, size=10),
    "value": np.random.uniform(10.0, 500.0, size=10)
})
df_head.head(0).to_csv(filename, index=False)

print("جاري توليد البيانات للوصول إلى الحجم المطلوب...")
while os.path.getsize(filename) / (1024**3) < target_size_gb:
    df_chunk = pd.DataFrame({
        "id": np.random.randint(1, 10**8, size=chunk_rows),
        "feature_1": np.random.randn(chunk_rows),
        "feature_2": np.random.randint(0, 100, size=chunk_rows),
        "value": np.random.uniform(10.0, 500.0, size=chunk_rows)
    })
    df_chunk.to_csv(filename, mode='a', header=False, index=False)

print(f"تم إنشاء الملف بنجاح! الحجم الحالي: {os.path.getsize(filename) / (1024**3):.2f} GB")

جاري توليد البيانات للوصول إلى الحجم المطلوب...
تم إنشاء الملف بنجاح! الحجم الحالي: 5.20 GB


In [ ]:
import pandas as pd
import psutil
import os
import time

def print_memory_usage():
    mem = psutil.virtual_memory()
    print(f"RAM المستهلكة: {mem.percent}% | المستغل فعلياً: {mem.used / (1024**3):.2f} GB")

print_memory_usage()

try:
    start_time = time.time()
    print("محاولة قراءة الملف دفعة واحدة عبر pd.read_csv()...")
    # إذا كان جهازك يحتوي على رام ضخمة جداً (أكثر من 32GB)، قد لا ينهار الجهاز فوراً،
    # لكن على الأجهزة القياسية (8GB - 16GB) سيحدث Crash أو MemoryError
    df = pd.read_csv("large_dataset.csv")
    print(f"تمت القراءة بنجاح في {time.time() - start_time:.2f} ثانية")
except MemoryError as e:
    print(f"\n[!] حدث خطأ نفاذ الذاكرة: MemoryError!")
    print("السبب: يحتاج Pandas إلى تحميل كامل البيانات وتحويلها إلى كائنات داخل RAM دفعة واحدة، مما يتجاوز السعة المتاحة.")
    print_memory_usage()

RAM المستهلكة: 42.3% | المستغل فعلياً: 5.05 GB
محاولة قراءة الملف دفعة واحدة عبر pd.read_csv()...


In [ ]:
import pandas as pd
import psutil
import time

start_time = time.time()
chunk_size = 100_000
total_sum = 0.0
total_count = 0

print("بدء معالجة البيانات باستخدام Chunking...")

# استخدام chunksize يقوم بإرجاع Iterator بدلاً من تحميل كل شيء في RAM
for chunk in pd.read_csv("large_dataset.csv", chunksize=chunk_size):
    total_sum += chunk["value"].sum()
    total_count += chunk["value"].count()

global_mean = total_sum / total_count
execution_time = time.time() - start_time

print(f"\nالنتيجة النهائية للمتوسط (Global Mean): {global_mean:.4f}")
print(f"الزمن المستغرق: {execution_time:.2f} ثانية")
print(f"استهلاك الذاكرة الحالي: {psutil.virtual_memory().percent}%")

In [ ]:
import dask.dataframe as dd
import time
import psutil

start_time = time.time()

# 1. قراءة البيانات (تعتمد على Lazy Evaluation - لا يتم تحميل البيانات فعلياً هنا)
ddf = dd.read_csv("large_dataset.csv")

# 2. بناء الرسم البياني للحسابات (Computation Graph)
mean_computation = ddf["value"].mean()

print("شرح Lazy Evaluation:")
print("Dask لم يقم بقراءة الملف بالكامل حتى الآن، بل قام فقط بتجهيز خطة التنفيذ (Task Graph).")

# 3. تشغيل الحسابات الفعلية وتوزيعها بالتوازي
result = mean_computation.compute()
execution_time = time.time() - start_time

print(f"\nالنتيجة عبر Dask: {result:.4f}")
print(f"الزمن المستغرق: {execution_time:.2f} ثانية")
print(f"استهلاك الذاكرة: {psutil.virtual_memory().percent}%")